# Bird Feeder AI: ViT-Small / ViT-Base / EfficientNet-Lite4 to Hailo HEF Conversion

Converts a fine-tuned bird species classifier (555 NABirds classes) from ONNX to Hailo HEF format for deployment on the Hailo-10H NPU (AI HAT+ 2).

Supports three models:
- **ViT-Small** (primary) — 224x224 input, requires mixed-precision quantization for self-attention layers
- **ViT-Base** (higher-capacity alternative) — 224x224 input, same mixed-precision scheme as ViT-Small (~4x params, higher accuracy)
- **EfficientNet-Lite4** (runner-up) — 300x300 input, standard INT8 quantization

**Before running, upload these files to a Google Drive folder called `hailo_convert/`:**
1. `hailo_dataflow_compiler-5.3.0-py3-none-linux_x86_64.whl` (from [Hailo Developer Zone](https://hailo.ai/developer-zone/software-downloads/))
2. Your ONNX model: `vit_small_birds.onnx`, `vit_base_birds.onnx`, and/or `efficientnet_lite4_birds.onnx` (from `models/onnx/`)
3. `calib_nabirds.zip` (calibration images from your NABirds dataset)

Your Drive folder should look like:
```
My Drive/
  hailo_convert/
    hailo_dataflow_compiler-5.3.0-py3-none-linux_x86_64.whl
    vit_small_birds.onnx           <- from: python -m src.training.export_onnx classifier --model vit_small
    vit_base_birds.onnx            <- (optional) from: python -m src.training.export_onnx classifier --model vit_base
    efficientnet_lite4_birds.onnx  <- (optional) from: python -m src.training.export_onnx classifier --model efficientnet_lite4
    calib_nabirds.zip
```


## Configuration

Choose which model to compile. Run this cell, then run all cells below.

In [ ]:
#@title Select Model to Compile { run: "auto" }
MODEL = "vit_small"  #@param ["vit_small", "vit_base", "efficientnet_lite4"]

# Model-specific settings (do not edit)
MODEL_CONFIGS = {
    "vit_small": {
        "onnx_file": "vit_small_birds.onnx",
        "input_size": 224,
        "hef_name": "vit_small_birds",
    },
    "vit_base": {
        "onnx_file": "vit_base_birds.onnx",
        "input_size": 224,
        "hef_name": "vit_base_birds",
    },
    "efficientnet_lite4": {
        "onnx_file": "efficientnet_lite4_birds.onnx",
        "input_size": 300,
        "hef_name": "efficientnet_lite4_birds",
    },
}

cfg = MODEL_CONFIGS[MODEL]
print(f"Model:      {MODEL}")
print(f"ONNX file:  {cfg['onnx_file']}")
print(f"Input size: {cfg['input_size']}x{cfg['input_size']}")
print(f"HEF output: {cfg['hef_name']}.hef")


## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
import glob
import os

drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/hailo_convert'
print('Files in hailo_convert/:')
!ls -lh {DRIVE_DIR}/

# Verify the selected ONNX model exists
onnx_path = os.path.join(DRIVE_DIR, cfg['onnx_file'])
if not os.path.exists(onnx_path):
    raise FileNotFoundError(
        f"\n\nONNX model not found: {cfg['onnx_file']}\n"
        f"Upload it to Google Drive at: {DRIVE_DIR}/{cfg['onnx_file']}\n"
        f"Generate it with: python -m src.training.export_onnx classifier --model {MODEL}"
    )
print(f"\nONNX model found: {cfg['onnx_file']}")

## Step 2: Install System Dependencies

In [ ]:
!sudo apt-get update -qq
!sudo apt-get install -y -qq python3-dev python3-distutils python3-tk libfuse2 graphviz libgraphviz-dev

## Step 3: Install the Hailo DFC

**First run:** installs the DFC into a virtualenv (~5 minutes).

**Subsequent runs:** if `hailo_env.tar.gz` exists in your Drive folder (`hailo_convert/`), the venv is restored from cache in under a minute — skipping the pip install entirely. Step 3b below exports the venv after a fresh install so this cache is built automatically.

In [ ]:
import shutil

VENV_TARBALL = f'{DRIVE_DIR}/hailo_env.tar.gz'

if os.path.exists(VENV_TARBALL):
    # Restore cached virtualenv — much faster than a fresh pip install
    size_mb = os.path.getsize(VENV_TARBALL) / (1024 * 1024)
    print(f'Found cached virtualenv: {VENV_TARBALL} ({size_mb:.1f} MB)')
    print('Extracting...')
    if os.path.isdir('/content/hailo_env'):
        shutil.rmtree('/content/hailo_env')
    !tar -xzf "{VENV_TARBALL}" -C /content
    print('Virtualenv restored from cache.')
else:
    # No cached venv — do a fresh install
    whl_files = glob.glob(f'{DRIVE_DIR}/hailo_dataflow_compiler*.whl')
    if not whl_files:
        raise FileNotFoundError(f'No DFC wheel found in {DRIVE_DIR}/')
    whl_file = whl_files[0]
    print(f'No cached venv found. Installing from: {whl_file}')

    # Install DFC in a virtualenv (required due to a known DFC packaging bug)
    !pip install -q virtualenv
    !virtualenv hailo_env
    !hailo_env/bin/pip install -q {whl_file}

    # Pin compatible numpy/scipy/Pillow versions
    !hailo_env/bin/pip install -q "numpy>=1.23,<2.0" "scipy>=1.10,<1.12" "Pillow>=9.0"

# ALWAYS install onnx-simplifier — required by DFC to simplify ViT's
# fused attention Mul ops. Idempotent: no-op if already installed.
# Without this, DFC fails to parse ViT with:
#   "ew mult layer ew_mult1 expects 2 inputs but found 1"
!hailo_env/bin/pip install -q onnx-simplifier

# Verify installation
!hailo_env/bin/python -c "from hailo_sdk_client import ClientRunner; print('DFC installed successfully')"
!hailo_env/bin/python -c "import onnxsim; print(f'onnx-simplifier installed: {onnxsim.__version__}')"

## Step 3b: Export venv to Drive (Cache for Future Runs)

Tars up the freshly-installed `hailo_env/` and saves it to your Drive folder. On the next Colab run, Step 3 will detect this tarball and restore the venv in under a minute instead of reinstalling.

Skipped automatically if the tarball already exists. Delete `hailo_convert/hailo_env.tar.gz` from Drive to force a re-export (e.g., after upgrading the DFC).

In [ ]:
VENV_TARBALL = f'{DRIVE_DIR}/hailo_env.tar.gz'

if os.path.exists(VENV_TARBALL):
    size_mb = os.path.getsize(VENV_TARBALL) / (1024 * 1024)
    print(f'Cached venv already exists on Drive: {VENV_TARBALL} ({size_mb:.1f} MB)')
    print('Skipping export. Delete the tarball from Drive to force a re-export.')
else:
    print('Exporting virtualenv to Drive (this takes a couple of minutes)...')
    !tar -czf "{VENV_TARBALL}" -C /content hailo_env
    size_mb = os.path.getsize(VENV_TARBALL) / (1024 * 1024)
    print(f'Exported: {VENV_TARBALL} ({size_mb:.1f} MB)')
    print('Future runs will restore the venv from this cache automatically.')

## Step 4: Set Up Files

In [ ]:
os.makedirs('/content/onnx', exist_ok=True)
os.makedirs('/content/hef', exist_ok=True)
os.makedirs('/content/calib_images', exist_ok=True)

# Copy ONNX model from Drive (faster than reading directly during compilation)
!cp "{DRIVE_DIR}/{cfg['onnx_file']}" /content/onnx/
print(f"ONNX model copied: {cfg['onnx_file']}")
!ls -lh /content/onnx/

# Extract calibration images
!unzip -q -o {DRIVE_DIR}/calib_nabirds.zip -d /content/calib_images
num = len(glob.glob('/content/calib_images/**/*.jpg', recursive=True) + 
          glob.glob('/content/calib_images/**/*.png', recursive=True))
print(f'Extracted {num} calibration images')

## Step 5: Write the Conversion Script

This generates the conversion Python script with the correct model-specific settings.

**Key differences between models:**
- **ViT-Small** requires a full `.alls` model script with mixed-precision quantization (16-bit for self-attention residual connections, 8-bit elsewhere) to maintain accuracy. Matches the [Hailo Model Zoo vit_small.alls](https://github.com/hailo-ai/hailo_model_zoo/blob/master/hailo_model_zoo/cfg/alls/generic/vit_small.alls).
- **ViT-Base** uses the same mixed-precision strategy but with the settings from [vit_base.alls](https://github.com/hailo-ai/hailo_model_zoo/blob/master/hailo_model_zoo/cfg/alls/generic/vit_base.alls) — smaller calibration batch (8 vs 16) and fewer resource-tuning knobs, since the Model Zoo leaves those at compiler defaults for ViT-Base.
- **EfficientNet-Lite4** uses standard INT8 quantization with just normalization.


In [ ]:
# Write model-specific conversion settings to a JSON file the script will read
import json

conversion_config = {
    "model": MODEL,
    "onnx_file": cfg["onnx_file"],
    "input_size": cfg["input_size"],
    "hef_name": cfg["hef_name"],
}

with open("/content/conversion_config.json", "w") as f:
    json.dump(conversion_config, f)

print(f"Config written for: {MODEL}")

In [ ]:
%%writefile /content/convert_hailo.py
"""ONNX to HEF conversion for bird species classifier (ViT-Small, ViT-Base, or EfficientNet-Lite4)."""

import json
import logging
import random
from pathlib import Path

import numpy as np
import onnx
from onnxsim import simplify
from PIL import Image
from hailo_sdk_client import ClientRunner

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

# Load configuration
with open("/content/conversion_config.json") as f:
    cfg = json.load(f)

MODEL = cfg["model"]
ONNX_FILE = cfg["onnx_file"]
INPUT_SIZE = cfg["input_size"]
HEF_NAME = cfg["hef_name"]


def simplify_onnx(input_path, output_path):
    """Pre-simplify the ONNX with onnx-simplifier.

    ViT models have fused attention operations (e.g., q = q * scale) that
    PyTorch exports as Mul nodes with one folded constant input. The Hailo DFC
    parser expects two explicit tensor inputs to ew_mult layers and fails with:
        "ew mult layer ew_mult1 expects 2 inputs but found 1"

    Running onnx-simplifier first normalizes these folded operations so the
    DFC can parse them correctly. This matches Hailo Model Zoo's preprocessing
    (their vit_small_patch16_224_ops17.onnx was pre-simplified).
    """
    logger.info(f"Simplifying ONNX: {input_path} -> {output_path}")
    model = onnx.load(input_path)
    simplified, check = simplify(model)
    if not check:
        raise RuntimeError("onnx-simplifier validation failed")
    onnx.save(simplified, output_path)
    logger.info(f"  Simplified ONNX saved ({Path(output_path).stat().st_size / 1024 / 1024:.1f} MB)")


def load_calibration_images(calib_dir, num_images=1024):
    """Load RAW (unnormalized) calibration images in NHWC format, values 0-255.

    Uses direct resize (not Resize+CenterCrop) to match production preprocessing.
    Production input is a tight YOLO crop where the bird fills the frame -
    center-cropping would discard discriminative features at the edges.
    """
    image_paths = sorted(
        list(Path(calib_dir).rglob('*.jpg'))
        + list(Path(calib_dir).rglob('*.jpeg'))
        + list(Path(calib_dir).rglob('*.png'))
    )
    if not image_paths:
        raise FileNotFoundError(f'No images found in {calib_dir}')

    random.seed(42)
    if len(image_paths) > num_images:
        image_paths = random.sample(image_paths, num_images)
    logger.info(f'Loading {len(image_paths)} calibration images')

    images = []
    for path in image_paths:
        try:
            img = Image.open(path).convert('RGB')
            # Direct resize to square - matches val/inference transforms
            img = img.resize((INPUT_SIZE, INPUT_SIZE), Image.BILINEAR)
            arr = np.array(img, dtype=np.float32)
            images.append(arr)
        except Exception as e:
            logger.warning(f'Skipping {path}: {e}')

    logger.info(f'Loaded {len(images)} images ({INPUT_SIZE}x{INPUT_SIZE}, NHWC, 0-255)')
    return np.array(images, dtype=np.float32)


def get_model_script_vit_small():
    """Generate .alls model script for ViT-Small.

    Based on the Hailo Model Zoo vit_small.alls:
    https://github.com/hailo-ai/hailo_model_zoo/blob/master/hailo_model_zoo/cfg/alls/generic/vit_small.alls

    Critical: ViT self-attention residual connections (ew_add layers) need 16-bit
    precision to maintain accuracy. Without this, quantization collapses the
    attention mechanism and accuracy drops severely.

    The layer prefix must match the model_name passed to translate_onnx_model.
    """
    return f"""norm_layer1 = normalization([127.5, 127.5, 127.5], [127.5, 127.5, 127.5])

context_switch_param(mode=enabled)
allocator_param(enable_partial_row_buffers=disabled)
allocator_param(automatic_reshapes=disabled)
buffers(conv1_s2d, conv1, 0, PARTIAL_ROW)
resources_param(strategy=greedy, max_compute_utilization=0.8, max_control_utilization=1.0, max_memory_utilization=0.8)

model_optimization_config(calibration, batch_size=16, calibset_size=1024)
post_quantization_optimization(finetune, policy=disabled)
pre_quantization_optimization(equalization, policy=enabled)
pre_quantization_optimization(ew_add_fusing, policy=disabled)
model_optimization_flavor(optimization_level=0, compression_level=0)

pre_quantization_optimization(matmul_correction, layers={{matmul*}}, correction_type=zp_comp_block)
model_optimization_config(negative_exponent, layers={{*}}, rank=0)

quantization_param({{{HEF_NAME}/ew_add*}}, precision_mode=a16_w16)
quantization_param({{{HEF_NAME}/ew_add1}}, precision_mode=a8_w8)
"""


def get_model_script_vit_base():
    """Generate .alls model script for ViT-Base.

    Based on the Hailo Model Zoo vit_base.alls:
    https://github.com/hailo-ai/hailo_model_zoo/blob/master/hailo_model_zoo/cfg/alls/generic/vit_base.alls

    Same mixed-precision strategy as ViT-Small (ew_add* -> a16_w16, ew_add1 -> a8_w8)
    because ViT-Base has the same self-attention residual structure. Differences
    from vit_small.alls:
      - batch_size=8 (ViT-Base is ~4x larger at 86.5M params, so smaller calib batch)
      - No context_switch_param / allocator_param / buffers tuning
      - max_control_utilization=0.85 (vs 1.0 for vit_small)
      - No post_quantization_optimization(finetune, ...) line

    The layer prefix must match the model_name passed to translate_onnx_model.
    """
    return f"""norm_layer1 = normalization([127.5, 127.5, 127.5], [127.5, 127.5, 127.5])
model_optimization_config(calibration, batch_size=8, calibset_size=1024)
pre_quantization_optimization(equalization, policy=enabled)
pre_quantization_optimization(ew_add_fusing, policy=disabled)
model_optimization_flavor(optimization_level=0, compression_level=0)
pre_quantization_optimization(matmul_correction, layers={{matmul*}}, correction_type=zp_comp_block)
model_optimization_config(negative_exponent, layers={{*}}, rank=0)
quantization_param({{{HEF_NAME}/ew_add*}}, precision_mode=a16_w16)
quantization_param({{{HEF_NAME}/ew_add1}}, precision_mode=a8_w8)

resources_param(strategy=greedy, max_compute_utilization=0.8, max_control_utilization=0.85, max_memory_utilization=0.8)
"""


def get_model_script_efficientnet_lite4():
    """Generate .alls model script for EfficientNet-Lite4.

    Our timm model (tf_efficientnet_lite4.in1k) uses mean=0.5, std=0.5
    normalization, which is [127.5, 127.5, 127.5] / [127.5, 127.5, 127.5]
    on the 0-255 scale. This matches our training transforms.

    Note: The Hailo Model Zoo's efficientnet_lite4.alls uses [127]/[128]
    because their model was TensorFlow-based. Ours is PyTorch/timm with
    different normalization, so we use [127.5]/[127.5] to match training.
    """
    return "normalization1 = normalization([127.5, 127.5, 127.5], [127.5, 127.5, 127.5])\n"


# ========== CONVERSION PIPELINE ==========

logger.info(f"Converting {MODEL} ({ONNX_FILE}) for hailo10h")
logger.info(f"Input size: {INPUT_SIZE}x{INPUT_SIZE}, Output: {HEF_NAME}.hef")

# --- Stage 0: Pre-simplify ONNX ---
# Required for ViT models so the DFC can parse fused attention Mul operations.
# Harmless (possibly beneficial) for EfficientNet-Lite4 too.
simplified_onnx = f'/content/onnx/{HEF_NAME}_simplified.onnx'
simplify_onnx(f'/content/onnx/{ONNX_FILE}', simplified_onnx)

# --- Stage 1: Parse ONNX ---
logger.info('Stage 1: Parsing ONNX model')
runner = ClientRunner(hw_arch='hailo10h')
runner.translate_onnx_model(
    simplified_onnx,
    HEF_NAME,
    start_node_names=['input'],
    end_node_names=['output'],
    net_input_shapes={'input': [1, 3, INPUT_SIZE, INPUT_SIZE]},
)
runner.save_har(f'/content/hef/{HEF_NAME}_parsed.har')
logger.info('  Parsed HAR saved')

# --- Stage 2: Load model script ---
if MODEL == 'vit_small':
    logger.info('Stage 2: Loading ViT-Small model script (mixed-precision quantization)')
    logger.info('  ew_add layers -> a16_w16 (16-bit for attention residuals)')
    logger.info('  ew_add1 -> a8_w8 (8-bit)')
    logger.info('  Normalization: [127.5, 127.5, 127.5] / [127.5, 127.5, 127.5]')
    model_script = get_model_script_vit_small()
elif MODEL == 'vit_base':
    logger.info('Stage 2: Loading ViT-Base model script (mixed-precision quantization)')
    logger.info('  ew_add layers -> a16_w16 (16-bit for attention residuals)')
    logger.info('  ew_add1 -> a8_w8 (8-bit)')
    logger.info('  Normalization: [127.5, 127.5, 127.5] / [127.5, 127.5, 127.5]')
    model_script = get_model_script_vit_base()
elif MODEL == 'efficientnet_lite4':
    logger.info('Stage 2: Loading EfficientNet-Lite4 model script (standard INT8)')
    logger.info('  Normalization: [127.5, 127.5, 127.5] / [127.5, 127.5, 127.5]')
    model_script = get_model_script_efficientnet_lite4()

runner.load_model_script(model_script)

# --- Stage 3: Load calibration data ---
logger.info('Stage 3: Loading calibration images')
calib_data = load_calibration_images('/content/calib_images', num_images=1024)

# --- Stage 4: Quantize ---
logger.info('Stage 4: Quantizing (this may take 10-30 minutes for ViT)...')
runner.optimize(calib_data)
runner.save_har(f'/content/hef/{HEF_NAME}_quantized.har')
logger.info('  Quantized HAR saved')

# --- Stage 5: Compile to HEF ---
logger.info('Stage 5: Compiling to HEF for hailo10h...')
hef_data = runner.compile()
hef_path = f'/content/hef/{HEF_NAME}.hef'
with open(hef_path, 'wb') as f:
    f.write(hef_data)

import os
size_mb = os.path.getsize(hef_path) / (1024 * 1024)
logger.info(f'Done! HEF saved: {hef_path} ({size_mb:.1f} MB)')


## Step 6: Run the Conversion

This runs the full pipeline: Parse ONNX -> Model script -> Calibrate -> Quantize -> Compile HEF.

**Expected time:**
- ViT-Small: ~15-30 minutes (mixed-precision quantization is slower)
- EfficientNet-Lite4: ~5-15 minutes

In [ ]:
%%time
!hailo_env/bin/python /content/convert_hailo.py

## Step 7: Copy HEF Back to Google Drive

In [ ]:
hef_name = cfg['hef_name']

print('Compiled HEF:')
!ls -lh /content/hef/*.hef

# Copy to Google Drive for persistent storage
!cp /content/hef/{hef_name}.hef {DRIVE_DIR}/
print(f'\nCopied to {DRIVE_DIR}/{hef_name}.hef')
print(f'Download it from Google Drive to your project: models/hef/{hef_name}.hef')

## Done!

The HEF file is saved to your Google Drive at `hailo_convert/<model>_birds.hef`.

Download it and place in your project:

```
models/hef/
  yolov11n.hef              <- pre-compiled from Hailo Model Zoo (bird detection)
  vit_small_birds.hef       <- just compiled (bird species classifier, 555 classes)
  vit_base_birds.hef        <- (optional) ViT-Base variant (same 555 classes, higher capacity)
```

Run on your Raspberry Pi:

```bash
python -m src.pipeline.pipeline --mode hailo
```

**Note:** The HEF has normalization baked in. The Hailo inference backend sends raw uint8 pixels (0-255) -- do NOT apply PyTorch-style normalization on the host side.

### Normalization values baked into each HEF

All three models use the same normalization because the timm models were all pretrained with `mean=0.5, std=0.5`:

| Model | Mean (0-255 scale) | Std (0-255 scale) | Matches training transforms.py |
|---|---|---|---|
| ViT-Small | [127.5, 127.5, 127.5] | [127.5, 127.5, 127.5] | Yes (0.5 * 255 = 127.5) |
| ViT-Base | [127.5, 127.5, 127.5] | [127.5, 127.5, 127.5] | Yes (0.5 * 255 = 127.5) |
| EfficientNet-Lite4 | [127.5, 127.5, 127.5] | [127.5, 127.5, 127.5] | Yes (0.5 * 255 = 127.5) |
